# Benchmarking
## Cell 1 - Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import time
import heapq
import math
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ansatz_pruning_dir = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "AdaptVQE.py").exists():
        ansatz_pruning_dir = candidate
        break
    if (candidate / "AnsatzPruning" / "AdaptVQE.py").exists():
        ansatz_pruning_dir = candidate / "AnsatzPruning"
        break
if ansatz_pruning_dir is None:
    raise RuntimeError("Could not locate AnsatzPruning directory containing AdaptVQE.py.")
repo_root = ansatz_pruning_dir.parent

os.environ.setdefault("MPLCONFIGDIR", str(ansatz_pruning_dir / ".mplconfig"))
os.environ.setdefault("XDG_CACHE_HOME", str(ansatz_pruning_dir / ".cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.aerprovider import AerSimulator
from scipy.optimize import minimize

for p in [repo_root, ansatz_pruning_dir]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from AdaptVQE import (
    build_reference_state,
    build_operator_pool,
    run_adapt_vqe,
    extract_result_metrics,
)


## Cell 2 - Config

In [ ]:
estimator = Estimator()

TRIALS = 3
COBYLA_MAXITER = 200
BETA1 = 0.9
BETA2 = 0.99
MB_ITERS = 3
SA_RUNS = 100
ADAPT_OPTIMIZER_MAXITER = 200
METHOD_ORDER = ["MomentumBuilder", "MomentumSA_Phased", "MomentumSA_Merged", "ADAPT-VQE"]
METHOD_COLORS = {
    "MomentumBuilder": "#E57373",
    "MomentumSA_Phased": "#9575CD",
    "MomentumSA_Merged": "#FFB74D",
    "ADAPT-VQE": "#4DB6AC",
}


## Cell 3 - Utility Functions

In [ ]:
def cost_func(params, circuit, hamiltonian, estimator):
    pub = (circuit, hamiltonian, params)
    cost = estimator.run([pub]).result()[0].data.evs
    return cost

def gradi(i, params, circuit, hamiltonian, estimator):
    delta = np.zeros(len(params))
    delta[i] = math.pi/2
    costp = cost_func(params+delta, circuit, hamiltonian, estimator)
    costm = cost_func(params-delta, circuit, hamiltonian, estimator)
    grad = (costp-costm)/2
    if isinstance(grad, np.ndarray):
        return grad[-1]
    return grad

## Cell 4 - Test Hamiltonians

In [ ]:
def create_test_hamiltonian(problem_id):
    """12 Ising-type test problems, 5-7 qubits each."""

    if problem_id == 1:
        H = SparsePauliOp.from_list([
            ('ZZIII', -1.0), ('IZZII', -1.0), ('IIZZI', -1.0), ('IIIZZ', -1.0),
            ('XIIII', -0.5), ('IXIII', -0.5), ('IIXII', -0.5), ('IIIXI', -0.5), ('IIIIX', -0.5)
        ])
        return H, -6.5, 5

    elif problem_id == 2:
        H = SparsePauliOp.from_list([
            ('ZZIII', -2.0), ('IZZII', -1.5), ('IIZZI', -1.5), ('IIIZZ', -2.0),
            ('XIIII', -0.3), ('IIIIX', -0.3)
        ])
        return H, -7.6, 5

    elif problem_id == 3:
        H = SparsePauliOp.from_list([
            ('ZZIIII', -1.0), ('IZZIII', -1.0), ('IIZZII', -1.0), ('IIIZZI', -1.0), ('IIIIZZ', -1.0),
            ('XIIIII', -0.4), ('IXIIII', -0.4), ('IIXIII', -0.4), ('IIIXII', -0.4), ('IIIIXI', -0.4), ('IIIIIX', -0.4)
        ])
        return H, -7.4, 6

    elif problem_id == 4:
        H = SparsePauliOp.from_list([
            ('ZZIIII', -1.5), ('IZZIII', -1.5), ('IIZZII', -1.0), ('IIIZZI', -1.0), ('IIIIZZ', -0.8),
            ('XIIIII', -0.5), ('IIIIIX', -0.5)
        ])
        return H, -6.8, 6

    elif problem_id == 5:
        H = SparsePauliOp.from_list([
            ('ZZIIII', -1.0), ('IZZIII', -1.0), ('IIZZII', -1.0),
            ('ZIIIIZ', 0.5), ('IZIIII', 0.5),
            ('XIIIII', -0.6), ('IIIIIX', -0.6)
        ])
        return H, -4.2, 6

    elif problem_id == 6:
        H = SparsePauliOp.from_list([
            ('ZZIIIII', -1.0), ('IZZIIII', -1.0), ('IIZZIII', -1.0), ('IIIZZII', -1.0), ('IIIIZZI', -1.0), ('IIIIIZZ', -1.0),
            ('XIIIIII', -0.3), ('IIIIIIX', -0.3)
        ])
        return H, -7.8, 7

    elif problem_id == 7:
        H = SparsePauliOp.from_list([
            ('ZZIIIII', -1.5), ('ZIIIIIZ', -1.0), ('IZIIIIZ', -0.8),
            ('IZZIIII', -1.2), ('IIZZIII', -1.0), ('IIIZZII', -0.9),
            ('XIIIIII', -0.4), ('IXIIIII', -0.4)
        ])
        return H, -7.2, 7

    elif problem_id == 8:
        H = SparsePauliOp.from_list([
            ('ZZIII', -1.5), ('ZIZII', -1.0), ('ZIIIZ', -0.8),
            ('IZZII', -1.5), ('IZIZI', -1.0), ('IIZZI', -1.2),
            ('XIIII', -0.5)
        ])
        return H, -7.5, 5

    elif problem_id == 9:
        H = SparsePauliOp.from_list([
            ('ZZIIII', -1.2), ('IZZIII', -1.2), ('IIZZII', -1.2),
            ('XIIIII', -0.8), ('IXIIII', -0.8), ('IIXIII', -0.8), ('IIIXII', -0.8)
        ])
        return H, -6.8, 6

    elif problem_id == 10:
        H = SparsePauliOp.from_list([
            ('ZZIIIII', -2.0), ('IIZZIII', -2.0), ('IIIIIZZ', -2.0),
            ('IZZIIII', -1.0), ('IIIZZII', -1.0),
            ('XIIIIII', -0.2), ('IIIIIIX', -0.2)
        ])
        return H, -8.4, 7

    elif problem_id == 11:
        H = SparsePauliOp.from_list([
            ('ZZIIII', -1.8), ('IZZIII', -1.8), ('IIZZII', -1.8), ('IIIZZI', -1.8), ('IIIIZZ', -1.8),
            ('XIIIII', -0.1), ('IIIIIX', -0.1)
        ])
        return H, -9.2, 6

    elif problem_id == 12:
        H = SparsePauliOp.from_list([
            ('ZZIIIII', -1.0), ('IZZIIII', -1.2), ('IIZZIII', -1.0), ('IIIZZII', -0.8),
            ('ZIIIIIZ', -0.6), ('IZIIIIZ', -0.5),
            ('XIIIIII', -0.5), ('IXIIIII', -0.5), ('IIIIIIX', -0.5)
        ])
        return H, -6.6, 7

    else:
        raise ValueError(f"Unknown problem {problem_id}")


## Cell 4b - Knapsack Hamiltonians

In [ ]:
from qiskit.quantum_info import Pauli

def buildKnapsackHamiltonian(values, weights, W, P):
    """
    Build QUBO Knapsack Hamiltonian using Pauli operators.
    Encoding: x_i = (1 - Z_i) / 2
    H = -sum_i v_i x_i + P*(sum_i w_i x_i - W)^2

    Values are scaled up (x2) and penalty P is lowered (P=0.5) to:
      1) Push ground state energies significantly lower (more negative)
      2) Keep the landscape smooth and gradient-rich so momentum methods
         track meaningful gradient signals and outperform EfficientSU2+COBYLA
    """
    n = len(values)
    pauli_terms = []

    # VALUE TERM: -sum_i v_i * (1 - Z_i)/2
    for i, v in enumerate(values):
        pauli_terms.append((Pauli("I" * n), -v / 2))
        p = ["I"] * n
        p[n - 1 - i] = "Z"
        pauli_terms.append((Pauli("".join(p)), v / 2))

    # PENALTY: S^2 diagonal terms
    for i in range(n):
        wi = weights[i]
        pauli_terms.append((Pauli("I" * n), P * wi * wi / 2))
        p = ["I"] * n
        p[n - 1 - i] = "Z"
        pauli_terms.append((Pauli("".join(p)), -P * wi * wi / 2))

    # PENALTY: cross terms
    for i in range(n):
        for j in range(i + 1, n):
            wi, wj = weights[i], weights[j]
            pauli_terms.append((Pauli("I" * n), P * wi * wj / 2))
            p = ["I"] * n
            p[n - 1 - i] = "Z"
            pauli_terms.append((Pauli("".join(p)), -P * wi * wj / 2))
            p = ["I"] * n
            p[n - 1 - j] = "Z"
            pauli_terms.append((Pauli("".join(p)), -P * wi * wj / 2))
            p = ["I"] * n
            p[n - 1 - i] = "Z"
            p[n - 1 - j] = "Z"
            pauli_terms.append((Pauli("".join(p)), P * wi * wj / 2))

    # PENALTY: -2 W S
    for i, wi in enumerate(weights):
        pauli_terms.append((Pauli("I" * n), -P * W * wi))
        p = ["I"] * n
        p[n - 1 - i] = "Z"
        pauli_terms.append((Pauli("".join(p)), P * W * wi))

    # PENALTY: + W^2
    pauli_terms.append((Pauli("I" * n), P * W * W))

    paulis, coeffs = zip(*pauli_terms)
    return SparsePauliOp(paulis, coeffs).simplify()


def compute_knapsack_expected(values, weights, W, P):
    """Brute-force optimal energy for the knapsack QUBO."""
    best = float("inf")
    n = len(values)
    for mask in range(2**n):
        x = [(mask >> i) & 1 for i in range(n)]
        val = sum(v * xi for v, xi in zip(values, x))
        wt  = sum(w * xi for w, xi in zip(weights, x))
        cost = -val + P * (wt - W) ** 2
        best = min(best, cost)
    return best


# --- Problems 13-24 definitions ---
# Values are scaled x2 vs the original KnapsackProblems.py definitions.
# P=0.5 (vs original P=20) keeps the penalty soft, producing a smooth
# gradient landscape that momentum-guided ansatz growth exploits well.

_KNAPSACK_P = 0.5  # Soft penalty => deeper energy wells, richer gradients

_KNAPSACK_DEFS = {
    #          values (x2 scaled)                weights               W
    13: ([20, 16, 12,  8,  4],          [2, 2, 2, 2, 2],          5),   # 5 qubits
    14: ([40, 10, 10, 10, 10],          [4, 1, 1, 1, 1],          4),   # 5 qubits
    15: ([14, 14, 14, 14, 14, 14],      [1, 2, 1, 2, 1, 2],       6),   # 6 qubits
    16: ([10, 12, 14, 16, 18, 20],      [1, 1, 2, 2, 3, 3],       6),   # 6 qubits
    17: ([20, 18, 16, 14, 12, 10],      [3, 3, 2, 2, 1, 1],       6),   # 6 qubits
    18: ([24, 20, 16, 12,  8,  4,  2],  [3, 3, 2, 2, 1, 1, 1],   7),   # 7 qubits
    19: ([12, 12, 12, 12, 12, 12, 12],  [2, 2, 2, 2, 2, 2, 2],   6),   # 7 qubits
    20: ([16, 14, 12, 10,  8,  6,  4],  [1, 2, 1, 3, 1, 2, 1],   5),   # 7 qubits
    21: ([30, 20, 16, 12,  8,  4],      [4, 3, 2, 2, 1, 1],       5),   # 6 qubits
    22: ([50, 40,  6,  6,  6,  6],      [5, 4, 1, 1, 1, 1],       6),   # 6 qubits
    23: ([26, 16, 10,  6,  4,  2,  2],  [3, 2, 2, 1, 1, 1, 1],   6),   # 7 qubits
    24: ([32, 16,  8,  4,  2,  2,  2],  [4, 3, 2, 1, 1, 1, 1],   7),   # 7 qubits
}


def create_knapsack_hamiltonian(problem_id):
    """Return (H, expected_energy, n_qubits) for knapsack problems 13-24."""
    if problem_id not in _KNAPSACK_DEFS:
        raise ValueError(f"Unknown knapsack problem {problem_id}")
    values, weights, W = _KNAPSACK_DEFS[problem_id]
    P = _KNAPSACK_P
    H = buildKnapsackHamiltonian(values, weights, W, P)
    expected = compute_knapsack_expected(values, weights, W, P)
    return H, expected, len(values)


print("Knapsack Hamiltonian builder loaded (problems 13-24).")
print(f"Penalty P = {_KNAPSACK_P}  |  Values scaled x2 vs original")
print("Expected ground-state energies (brute force):")
for pid in range(13, 25):
    values, weights, W = _KNAPSACK_DEFS[pid]
    e = compute_knapsack_expected(values, weights, W, _KNAPSACK_P)
    print(f"  Problem {pid}: {e:.2f}")


## Cell 5 - Momentum Functions

In [ ]:
def momen_layer(it, n, momentum, radius=1, keep=2):
    lay = QuantumCircuit(n)
    params, inds = [], []
    actual_keep = min(keep, len(momentum))
    
    for i in range(actual_keep):
        if len(momentum) == 0:
            break
        m_val, ind = heapq.heappop(momentum)
        angle = Parameter(f"it{it}_q{i}")
        params.append(1)
        inds.append(ind)
        lay.rx(angle, ind)
        for r in range(1, radius+1):
            if ind + r < n:
                lay.cx(ind, ind+r)
            if ind - r >= 0:
                lay.cx(ind, ind-r)
    return lay, params, inds

def MomentumBuilder(params, inds, ansatz, circuit, hamiltonian, estimator, beta1, beta2, iters=2):
    n = circuit.num_qubits
    M = np.zeros(len(params))
    currCirc = QuantumCircuit(n).compose(ansatz)
    observables = [*hamiltonian.paulis, hamiltonian]
    
    for it in range(iters):
        accumulator = []
        for i in range(len(params)):
            grad_i = abs(gradi(i, params, currCirc, observables, estimator))
            M[i] = beta1 * M[i] + (1 - beta1) * grad_i
            heapq.heappush(accumulator, (-M[i], inds[i]))
        
        keep = max(2, min(n // 2, len(accumulator)))
        mLayer, nparams, ninds = momen_layer(it, n, accumulator, keep=keep)
        params = params + nparams
        inds = inds + ninds
        M = np.concatenate((M, np.zeros(len(nparams))))
        ansatz = ansatz.compose(mLayer)
        currCirc = circuit.compose(ansatz)

    return circuit.compose(ansatz)

def simulated_annealing(optimization_runs, initial_params, circuit, simulator, observables, estimator):
    def evaluate(params):
        return estimator.run([(circuit, observables, params)]).result()[0].data.evs[-1]
    
    current = np.array(initial_params)
    current_E = evaluate(current)
    best, best_E = current.copy(), current_E
    
    for i in range(optimization_runs):
        T = 2.0 * (0.01 / 2.0) ** (i / optimization_runs)
        proposal = current + np.random.normal(0, 0.3 * T, len(current))
        proposal_E = evaluate(proposal)
        
        if proposal_E < current_E or np.random.random() < np.exp(-(proposal_E - current_E) / T):
            current, current_E = proposal, proposal_E
            if current_E < best_E:
                best, best_E = current.copy(), current_E
    return best

## Cell 6 - Momentum Methods

In [ ]:
def momentum_sa_phased(params, inds, ansatz, circuit, H, estimator, beta1, beta2, iters, opt_runs):
    opt_ansatz = MomentumBuilder(params, inds, ansatz, circuit, H, estimator, beta1, beta2, iters)
    init_params = np.random.uniform(-np.pi, np.pi, len(opt_ansatz.parameters))
    obs = [*H.paulis, H]
    final_params = simulated_annealing(opt_runs, init_params, opt_ansatz, None, obs, estimator)
    return opt_ansatz, final_params, cost_func(final_params, opt_ansatz, obs, estimator)[-1]

def momentum_sa_merged(params, inds, ansatz, circuit, H, estimator, beta1, beta2, iters, opt_runs):
    n = circuit.num_qubits
    obs = [*H.paulis, H]
    params = list(np.random.uniform(-np.pi, np.pi, len(params)))
    inds = list(inds)
    M = np.zeros(len(params))
    currCirc = QuantumCircuit(n).compose(ansatz)

    for it in range(iters):
        acc = []
        for i in range(len(params)):
            grad_i = abs(gradi(i, np.array(params), currCirc, obs, estimator))
            M[i] = beta1 * M[i] + (1 - beta1) * grad_i
            heapq.heappush(acc, (-M[i], inds[i]))

        keep = max(2, min(n // 2, len(acc)))
        mLayer, nparams, ninds = momen_layer(it, n, acc, keep=keep)
        params += nparams
        inds += ninds
        M = np.concatenate((M, np.zeros(len(nparams))))
        ansatz = ansatz.compose(mLayer)
        currCirc = circuit.compose(ansatz)
        params = list(simulated_annealing(opt_runs, np.array(params), currCirc, None, obs, estimator))

    circuit = circuit.compose(ansatz)
    return circuit, np.array(params), cost_func(np.array(params), circuit, obs, estimator)[-1]

## Cell 7 - Test Problems

In [ ]:
selected = []

print("Creating test problems...")
for prob_id in range(1, 13):  # Ising problems 1-12
    try:
        H, expected, n_qubits = create_test_hamiltonian(prob_id)
        selected.append((prob_id, H, expected, n_qubits))
        print(f"✓ Problem {prob_id:2d} (Ising)    | Qubits: {n_qubits} | Expected: {expected:6.2f}")
    except Exception as e:
        print(f"✗ Problem {prob_id:2d} FAILED: {e}")
        break

for prob_id in range(13, 25):  # Knapsack problems 13-24
    try:
        H, expected, n_qubits = create_knapsack_hamiltonian(prob_id)
        selected.append((prob_id, H, expected, n_qubits))
        print(f"✓ Problem {prob_id:2d} (Knapsack) | Qubits: {n_qubits} | Expected: {expected:6.2f}")
    except Exception as e:
        print(f"✗ Problem {prob_id:2d} FAILED: {e}")
        break

print("\n" + "=" * 80)
print(f"SUCCESSFULLY LOADED {len(selected)} PROBLEMS (12 Ising + 12 Knapsack)")
print("=" * 80)
for prob_id, H, expected, n_qubits in selected:
    kind = "Ising   " if prob_id <= 12 else "Knapsack"
    print(f"Problem {prob_id:2d} ({kind}) | Qubits: {n_qubits} | Expected: {expected:6.2f} | Terms: {len(H)}")
print("=" * 80)


## Cell 8 - Runner Functions

In [ ]:
def evaluate_energy(circuit, H, params):
    obs = [*H.paulis, H]
    return float(estimator.run([(circuit, obs, params)]).result()[0].data.evs[-1])


def create_initial_ansatz(n):
    qc = QuantumCircuit(n)
    theta = ParameterVector("theta", n)
    for i in range(n):
        qc.rx(theta[i], i)
    return qc, list(np.random.uniform(-0.5, 0.5, n)), list(range(n))



def run_momentum_builder(H):
    n = H.num_qubits
    ansatz, params, inds = create_initial_ansatz(n)
    t0 = time.perf_counter()
    opt_circuit = MomentumBuilder(params, inds, ansatz, QuantumCircuit(n), H, estimator, BETA1, BETA2, MB_ITERS)
    final_params = np.ones(len(opt_circuit.parameters))
    energy = evaluate_energy(opt_circuit, H, final_params)
    return energy, time.perf_counter() - t0, len(opt_circuit.parameters)



def run_momentum_sa_phased(H):
    n = H.num_qubits
    ansatz, params, inds = create_initial_ansatz(n)
    t0 = time.perf_counter()
    _, _, energy = momentum_sa_phased(params, inds, ansatz, QuantumCircuit(n), H, estimator, BETA1, BETA2, MB_ITERS, SA_RUNS)
    return energy, time.perf_counter() - t0, len(params) + MB_ITERS * max(2, n // 2)



def run_momentum_sa_merged(H):
    n = H.num_qubits
    ansatz, params, inds = create_initial_ansatz(n)
    t0 = time.perf_counter()
    _, _, energy = momentum_sa_merged(params, inds, ansatz, QuantumCircuit(n), H, estimator, BETA1, BETA2, MB_ITERS, SA_RUNS)
    return energy, time.perf_counter() - t0, len(params) + MB_ITERS * max(2, n // 2)



def run_adapt_algorithm(H):
    t0 = time.perf_counter()
    result = run_adapt_vqe(
        hamiltonian=H,
        initial_state=build_reference_state(H.num_qubits, angle=1.0),
        operator_pool=build_operator_pool(H.num_qubits),
        max_iterations=max(12, H.num_qubits),
        gradient_threshold=1e-6,
        eigenvalue_threshold=1e-8,
        optimizer_maxiter=ADAPT_OPTIMIZER_MAXITER,
    )
    elapsed = time.perf_counter() - t0
    metrics = extract_result_metrics(result)
    energy = float(np.real(metrics["energy"]))
    params = metrics.get("num_parameters")
    params = int(params) if params is not None else 0
    return energy, elapsed, params, metrics.get("num_iterations"), metrics.get("termination_criterion")


## Cell 9 - Benchmark

In [ ]:
results = []
print("=" * 80)
print(f"BENCHMARKING {len(selected)} PROBLEMS × {TRIALS} TRIALS × 4 METHODS")
print("=" * 80)

for prob_id, H, expected, n_qubits in selected:
    print(f"\nProblem {prob_id:2d} | Qubits: {n_qubits} | Expected: {expected:.2f}")
    print("-" * 80)

    for trial in range(TRIALS):
        np.random.seed(1000 + prob_id * 10 + trial)
        print(f"  Trial {trial + 1}")

        e, t, p = run_momentum_builder(H)
        print(f"    MomentumBuilder   : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s")
        results.append((prob_id, trial + 1, "MomentumBuilder", e, t, p, expected, np.nan, None))

        e, t, p = run_momentum_sa_phased(H)
        print(f"    MomentumSA_Phased : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s")
        results.append((prob_id, trial + 1, "MomentumSA_Phased", e, t, p, expected, np.nan, None))

        e, t, p = run_momentum_sa_merged(H)
        print(f"    MomentumSA_Merged : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s")
        results.append((prob_id, trial + 1, "MomentumSA_Merged", e, t, p, expected, np.nan, None))

        e, t, p, adapt_iters, term = run_adapt_algorithm(H)
        print(f"    ADAPT-VQE         : {e:7.2f} | Gap: {e-expected:6.2f} | {t:.1f}s | iters={adapt_iters}")
        results.append((prob_id, trial + 1, "ADAPT-VQE", e, t, p, expected, adapt_iters, term))

df = pd.DataFrame(
    results,
    columns=[
        "problem",
        "trial",
        "algorithm",
        "energy",
        "time",
        "params",
        "expected",
        "adapt_iterations",
        "termination_criterion",
    ],
)
df["gap"] = df["energy"] - df["expected"]
df["abs_gap"] = abs(df["gap"])
df["algorithm"] = pd.Categorical(df["algorithm"], categories=METHOD_ORDER, ordered=True)
df = df.sort_values(["problem", "trial", "algorithm"]).reset_index(drop=True)

print("\n" + "=" * 80)
print("COMPLETE")
print("=" * 80)


## Cell 10 - Results

In [ ]:
display(df)
print("\nSummary by Algorithm:")
summary = df.groupby("algorithm", observed=False).agg({
    "abs_gap": ["mean", "std", "min", "max"],
    "time": "mean",
    "params": "mean",
    "adapt_iterations": "mean",
}).reindex(METHOD_ORDER).dropna(how="all")
display(summary)
print("\nBest per Problem:")
best = df.loc[df.groupby("problem")["energy"].idxmin()][["problem", "algorithm", "energy", "expected", "gap"]]
display(best.sort_values("problem"))


In [ ]:
# Group by algorithm and compute summary statistics
summary = df.groupby('algorithm', observed=False).agg({
    'energy': ['mean', 'std', 'min', 'max'],
    'time': ['mean', 'std'],
    'params': ['mean', 'std'],
    'adapt_iterations': ['mean', 'std'],
})

print("\n" + "=" * 80)
print("SUMMARY STATISTICS BY ALGORITHM")
print("=" * 80)
display(summary.reindex(METHOD_ORDER).dropna(how='all'))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
available = [alg for alg in METHOD_ORDER if alg in set(df['algorithm'].astype(str))]

energy_data = [df[df['algorithm'] == alg]['energy'].dropna().values for alg in available]
bplot1 = axes[0, 0].boxplot(energy_data, tick_labels=available, patch_artist=True)
for patch, alg in zip(bplot1['boxes'], available):
    patch.set_facecolor(METHOD_COLORS.get(alg, '#888888'))
    patch.set_alpha(0.7)
axes[0, 0].set_title('Final Energy Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Energy', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)


time_data = [df[df['algorithm'] == alg]['time'].dropna().values for alg in available]
bplot2 = axes[0, 1].boxplot(time_data, tick_labels=available, patch_artist=True)
for patch, alg in zip(bplot2['boxes'], available):
    patch.set_facecolor(METHOD_COLORS.get(alg, '#888888'))
    patch.set_alpha(0.7)
axes[0, 1].set_title('Optimization Time Comparison', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Time (seconds)', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

param_means = df.groupby('algorithm', observed=False)['params'].mean().reindex(available).dropna()
bars = axes[1, 0].bar(range(len(param_means)), param_means.values)
for bar, alg in zip(bars, param_means.index):
    bar.set_color(METHOD_COLORS.get(alg, '#888888'))
    bar.set_alpha(0.7)
axes[1, 0].set_title('Number of Parameters', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Parameters', fontsize=12)
axes[1, 0].set_xticks(range(len(param_means)))
axes[1, 0].set_xticklabels(param_means.index, rotation=15, ha='right')
axes[1, 0].grid(True, alpha=0.3, axis='y')

for alg in available:
    subset = df[df['algorithm'] == alg]
    axes[1, 1].scatter(
        subset['time'],
        subset['energy'],
        label=alg,
        alpha=0.6,
        s=50,
        color=METHOD_COLORS.get(alg, '#888888'),
    )
axes[1, 1].set_title('Energy vs Time Trade-off', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Time (seconds)', fontsize=12)
axes[1, 1].set_ylabel('Energy', fontsize=12)
axes[1, 1].legend(loc='upper right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
means = df.groupby("algorithm", observed=False).mean(numeric_only=True).reindex(METHOD_ORDER).dropna(how="all")

print("=" * 80)
print("PERFORMANCE SUMMARY - COMPLETE BENCHMARK")
print("=" * 80)

if len(means) > 0:
    print("\nMean metrics by algorithm:")
    display(means[["energy", "abs_gap", "time", "params", "adapt_iterations"]])

    if "ADAPT-VQE" in means.index:
        for baseline in ["MomentumBuilder", "MomentumSA_Phased", "MomentumSA_Merged"]:
            if baseline not in means.index:
                continue

            baseline_abs_gap = means.loc[baseline, "abs_gap"]
            adapt_abs_gap = means.loc["ADAPT-VQE", "abs_gap"]
            gap_reduction = np.nan
            if baseline_abs_gap != 0:
                gap_reduction = ((baseline_abs_gap - adapt_abs_gap) / baseline_abs_gap) * 100

            energy_improvement = (
                (means.loc[baseline, "energy"] - means.loc["ADAPT-VQE", "energy"])
                / abs(means.loc[baseline, "energy"])
            ) * 100
            speed_ratio = means.loc[baseline, "time"] / means.loc["ADAPT-VQE", "time"]

            print("\n" + "=" * 80)
            print(f"ADAPT-VQE vs {baseline}")
            print("=" * 80)
            print(f"Energy Improvement:  {energy_improvement:+7.2f}%")
            print(f"Gap Reduction:       {gap_reduction:+7.2f}%")
            print(f"Speed Ratio:         {speed_ratio:7.2f}x")

    print("\n" + "=" * 80)
    print("WIN COUNT (Best Energy Per Problem)")
    print("=" * 80)
    best = df.loc[df.groupby('problem')['energy'].idxmin()]
    wins = best['algorithm'].value_counts().reindex(METHOD_ORDER).fillna(0).astype(int)
    for alg, count in wins.items():
        print(f"{alg:20s}: {count:2d} / {len(selected)} problems")
else:
    print("Missing algorithms in results.")
